# Environment Setup

In [8]:
!pip install transformers datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.0 MB/s eta 0:00:00


#Imports

In [48]:
import pandas as pd
import numpy as np
import evaluate
from datasets import DatasetDict, Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import TrainingArguments
from transformers import AutoModelForSequenceClassification
from transformers import Trainer
from transformers import pipeline

# Data Loading and Preparation

In [49]:
df = pd.read_csv('B2W-Reviews01.csv')
df = df[['review_text', 'overall_rating']].copy()
df['overall_rating'] = df['overall_rating'].astype(int)
df['label'] = df['overall_rating'] - 1
df.rename(columns={'review_text': 'text'}, inplace=True)
df = df[['text', 'label']]

<ipython-input-49-1804200352>:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('B2W-Reviews01.csv')


# Convert to Hugging Face Format and Split the Data

In [50]:
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2, seed=42)

dataset_dict = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test']
})

# Tokenization

In [38]:
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["text"], truncation=True)

tokenized_datasets = dataset_dict.map(tokenize_function, batched=True, load_from_cache_file=False)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/20649 [00:00<?, ? examples/s]

Map:   0%|          | 0/5163 [00:00<?, ? examples/s]

# Model Definition

In [41]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=5)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Training Setup

In [40]:
output_dir = "./bert-B2W-Reviews01"
training_args = TrainingArguments(
  output_dir=output_dir,
  num_train_epochs=3,  # Number of training epochs
  per_device_train_batch_size=8,  # Batch size per GPU
  per_device_eval_batch_size=8,   # Batch size for evaluation per GPU
  weight_decay=0.01,   # Strength of weight decay
  logging_dir="./logs",   # Directory for storing logs
  logging_steps=100,   # Log every N steps
  eval_strategy="steps",   # Evaluation strategy during training
  eval_steps=200,   # Run evaluation every N steps
  save_total_limit=2,   # Only save the last N checkpoints
  save_steps=200,   # Save checkpoint every N steps
  load_best_model_at_end=True,   # Load the best model at the end of training
  metric_for_best_model="accuracy",   # Metric to use for the best model
)

In [42]:
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

<ipython-input-42-265344811>:8: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# Running Training and Evaluation

In [43]:
trainer.train()

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: stevansouza2003 (stevansouza2003-universidade-federal-de-itajub-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Accuracy
200,0.744300,0.676760,0.724579
400,0.655300,0.740988,0.747821
600,0.646500,0.570358,0.796242
800,0.602800,0.556052,0.805927
1000,0.576300,0.519105,0.816192
1200,0.506900,0.582298,0.818516
1400,0.553400,0.514028,0.819485
1600,0.496500,0.524697,0.814255
1800,0.517200,0.560229,0.809413
2000,0.547500,0.493766,0.822971


TrainOutput(global_step=7746, training_loss=0.4797718167274213, metrics={'train_runtime': 5359.597, 'train_samples_per_second': 11.558, 'train_steps_per_second': 1.445, 'total_flos': 3970406518545780.0, 'train_loss': 0.4797718167274213, 'epoch': 3.0})

In [44]:
print("\nEvaluation results on the test set:")
print(trainer.evaluate())


Evaluation results on the test set:


{'eval_loss': 0.4536758065223694, 'eval_accuracy': 0.84214603912454, 'eval_runtime': 38.7322, 'eval_samples_per_second': 133.3, 'eval_steps_per_second': 16.679, 'epoch': 3.0}


# Save the Model and Use for Prediction

In [47]:
trainer.save_model("./modelo_final_b2w_reviews")
tokenizer.save_pretrained("./modelo_final_b2w_reviews")

pipe = pipeline("text-classification", model="./modelo_final_b2w_reviews")

review_example = "The product is excellent, it exceeded my expectations! Arrived very quickly."
result = pipe(review_example)

print(f"\nPrediction example for the phrase: '{review_example}'")
print(result)

Device set to use cuda:0



Prediction example for the phrase: 'The product is excellent, it exceeded my expectations! Arrived very quickly.'
[{'label': 'LABEL_2', 'score': 0.9775276780128479}]
